In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 171 (delta 64), reused 146 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 19.99 MiB | 21.98 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os
import time
import jax
import jax.numpy as jnp

from core import Pars2d, Equation2d
from solver import Solver2d, FastSolver2d
from boundaries import periodic_bc_2d
from equations import make_euler_riemann_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod

In [12]:
def run_gpu_benchmark():
    print(f"JAX Device(s): {jax.devices()}")
    J = 200

    pars = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.4, Jx=J, Jy=J, cfl=0.475, scheme="sd2"
    )

    eqn = make_euler_riemann_2d()
    solver = FastSolver2d(pars, eqn, limiter_name="minmod")

    print("--- Прогрев JAX (Компиляция XLA) ---")
    # Прогреваем ТОЛЬКО первый шаг решателя (именно он содержит всю тяжелую математику)
    x_1d = jnp.linspace(pars.x_init + pars.dx / 2, pars.x_final - pars.dx / 2, pars.Jx)
    y_1d = jnp.linspace(pars.y_init + pars.dy / 2, pars.y_final - pars.dy / 2, pars.Jy)
    X, Y = jnp.meshgrid(x_1d, y_1d, indexing='ij')
    test_u = eqn.initial_data(X, Y)
    _ = solver.update_step_jit(0.0, test_u, 0.001).block_until_ready()
    print("Прогрев завершен.\n")

    print(f"--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, {J}x{J}) ---")
    t0 = time.time()
    results = solver.solve()
    results['u'][-1].block_until_ready()
    t1 = time.time()

    print(f"\n[GPU JAX] Чистое время выполнения: {t1 - t0:.4f} секунд")

if __name__ == "__main__":
    jax.config.update("jax_enable_x64", True)
    run_gpu_benchmark()

JAX Device(s): [CudaDevice(id=0)]
--- Прогрев JAX (Компиляция XLA) ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, 200x200) ---
Starting 2D simulation: Euler 2D (Riemann)
Grid: 200x200, Scheme: SD2/minmod
t = 0.4000 / 0.4000
Done! Wall time: 1.39 s

[GPU JAX] Чистое время выполнения: 1.4044 секунд
